# Laboratorio 11 — Provocar una fuga de información y corregirla

Un flujo de trabajo razonable sobre el registro de mantenimiento industrial ya auditado
alcanza un AUC de $1{,}0000$ sobre el conjunto de prueba. El laboratorio empieza
reproduciendo ese resultado y termina explicando por qué es una mala noticia.

El recorrido tiene tres tramos y un cierre:

1. **Provocar la fuga.** Se reproduce el flujo, se mide el número inflado y se identifica la
   columna culpable **leyendo el código, no la salida**.
2. **Medir dos fugas más** sobre el mismo conjunto: una que se corrige en una línea, y otra
   que no infla el número sino que **invierte la decisión** sobre un atributo.
3. **Corregirla por estructura.** Se arma el flujo como `Pipeline` con `ColumnTransformer` y
   se comprueba que la fuga dejó de ser escribible.
4. **El cierre** diagnostica el mecanismo de los tres faltantes del conjunto y mide qué le
   hace a los datos la imputación de uso más corriente.

**El entregable es la decisión de la sección 5**: qué flujo va a producción, con qué
columnas y con qué supuesto declarado. No es una función que pasa un test, es una conclusión
justificada con las cifras que el propio notebook produce.

Las celdas que contienen `raise NotImplementedError` deben completarse.

## 0. Preparación

El conjunto es `data/mantenimiento_industrial.csv`, el mismo registro de mantenimiento
industrial que ya fue auditado: $4237$ filas, 14 columnas y la respuesta binaria `fallo`. Lo
produce un script del curso con semilla fija, de modo que todos trabajan sobre exactamente
las mismas filas.

Dos rasgos del conjunto importan acá y ya se conocen de la auditoría: tiene **faltantes en
tres columnas numéricas**, inyectados con mecanismos distintos, y tiene **centinelas** —
temperaturas cargadas como `999` y presiones en `-1`— que no son mediciones posibles.

La celda de carga busca el archivo en las ubicaciones habituales. En Google Colab, si no lo
encuentra, abre el diálogo de subida.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120)
np.set_printoptions(precision=4, suppress=True)

# Paleta del curso
AZUL, TURQUESA, NARANJA = "#19326E", "#50ACB0", "#CD742A"
VIOLETA, OLIVA, GRIS = "#A3477F", "#89A943", "#9A9A9A"

SEMILLA = 42

In [ ]:
from pathlib import Path
import urllib.request

# Carga de datos del curso. No hace falta modificar nada.

URL_DATOS = "https://raw.githubusercontent.com/gustavovazquez/datasets/main/"


def ruta_del_dato(nombre: str) -> str:
    """Devuelve algo que `pd.read_csv` pueda leer, este donde este el notebook.

    Busca el archivo junto al notebook y en las carpetas `data/` de hasta tres
    niveles superiores; si no esta, usa la copia publicada en linea; y si
    tampoco hay red, en Colab ofrece subirlo a mano.
    """
    candidatas = [Path(nombre), Path("data") / nombre]
    for arriba in ("..", "../..", "../../.."):
        candidatas += [Path(arriba) / nombre, Path(arriba) / "data" / nombre]
    for ruta in candidatas:
        if ruta.exists():
            print(f"Cargado desde {ruta}")
            return str(ruta)

    url = URL_DATOS + nombre
    try:
        urllib.request.urlopen(url, timeout=15)
        print(f"No esta en disco; se descarga de {url}")
        return url
    except Exception:
        pass

    try:
        from google.colab import files  # type: ignore
    except ImportError as exc:
        raise FileNotFoundError(
            f"No se encontro {nombre}. Dejarlo en una carpeta 'data/' junto al notebook."
        ) from exc

    print(f"Falta {nombre}: seleccionar el archivo para subirlo.")
    files.upload()
    return nombre

In [ ]:
df = pd.read_csv(ruta_del_dato("mantenimiento_industrial.csv"))
y = df["fallo"].to_numpy()

# Los siete atributos numéricos de sensor y las tres categóricas de baja cardinalidad.
# `id_maquina` queda aparte: tiene 180 niveles y es el tema de la sección 2.2.
NUMERICAS = ["antiguedad_meses", "horas_operacion", "n_mantenimientos_previos",
             "temperatura_c", "vibracion_mm_s", "presion_bar", "consumo_kwh"]
CATEGORICAS = ["planta", "modelo_equipo", "turno"]

print(f"{df.shape[0]} registros, {df.shape[1]} columnas")
print(f"fallos: {y.sum()} ({100 * y.mean():.2f} %)")
print(f"filas duplicadas, ignorando id_registro: "
      f"{df.drop(columns=['id_registro']).duplicated().sum()}")
print("\nfaltantes por columna:")
print(df.isna().sum().loc[lambda s: s > 0].to_string())
print(f"\ncentinelas: temperatura_c = 999 en {int((df.temperatura_c == 999).sum())} filas · "
      f"presion_bar = -1 en {int((df.presion_bar == -1).sum())} filas")
print(f"\ncosto_reparacion_usd informado en {df.costo_reparacion_usd.notna().sum()} filas, "
      f"de las cuales fallo = 1 en "
      f"{int(df.loc[df.costo_reparacion_usd.notna(), 'fallo'].sum())}")

La última línea de esa salida conviene retenerla, porque la sección 1 la va a necesitar.

---

## 1. El modelo perfecto que no sirve

El flujo que sigue es el que escribiría cualquiera. Se toman los siete atributos numéricos
más `costo_reparacion_usd`, se parte en entrenamiento y prueba de forma estratificada, se
imputan los faltantes por la mediana, se estandariza y se ajusta un bosque aleatorio.

`costo_reparacion_usd` está informado solo cuando hubo una reparación. Para las demás filas
se lo completa con cero, con un argumento impecable: **si no hubo reparación, el costo fue
cero.**

**La celda viene entera y no hay que escribirla.** Lo que se pide después no es escribir el
flujo: es revisarlo.

In [ ]:
ATRIBUTOS = NUMERICAS + ["costo_reparacion_usd"]


def preparar(bloque: pd.DataFrame) -> np.ndarray:
    """Selecciona las columnas del modelo y completa el costo ausente con cero."""
    tabla = bloque[ATRIBUTOS].copy()
    tabla["costo_reparacion_usd"] = tabla["costo_reparacion_usd"].fillna(0.0)
    return tabla.to_numpy(dtype=float)


ent, pru, y_ent, y_pru = train_test_split(
    df, y, test_size=0.30, random_state=SEMILLA, stratify=y)

A, B = preparar(ent), preparar(pru)
imputador = SimpleImputer(strategy="median").fit(A)
escalador = StandardScaler().fit(imputador.transform(A))


def transformar(M: np.ndarray) -> np.ndarray:
    """Aplica al bloque las dos transformaciones ya ajustadas sobre entrenamiento."""
    return escalador.transform(imputador.transform(M))


bosque = RandomForestClassifier(n_estimators=300, random_state=SEMILLA,
                                n_jobs=-1).fit(transformar(A), y_ent)

auc_validacion = roc_auc_score(y_pru, bosque.predict_proba(transformar(B))[:, 1])
print(f"entrenamiento {len(y_ent)} filas · prueba {len(y_pru)} filas")
print(f"AUC sobre el conjunto de prueba: {auc_validacion:.4f}")

### DECIDE

El AUC es exactamente $1{,}0000$: el modelo ordena sin un solo error las $1272$
observaciones de prueba. Ningún fenómeno industrial se predice sin error a partir de siete
sensores, de modo que la hipótesis razonable no es que el modelo sea brillante.

**Corresponde responder antes de ejecutar ninguna celda más**, y sobre el código de arriba,
no sobre su salida. Las cuatro preguntas de revisión son las de la exposición:

1. ¿Qué se ejecuta **antes** de la línea que parte los datos? ¿Hay algún `fit` o
   `fit_transform` ahí?
2. Para cada columna: **¿en qué momento se registra su valor?** Es la única de las cuatro que
   no se responde leyendo código.
3. ¿Hay un orden temporal que la partición aleatoria esté ignorando?
4. ¿Hay observaciones repetidas o agrupadas?

**Con eso, decidir:** cuál de las cuatro formas de fuga está presente, qué columna la
produce, y por qué completarla con cero **no** la neutralizó.

**Respuesta:**

1. Antes de `train_test_split` solo se cargan los datos, se separa `y` y se define la lista de atributos. No hay ningún `fit` ni `fit_transform`, por lo que no aparece una fuga de preprocesamiento en ese tramo.
2. Los sensores y datos de la máquina existen antes de la predicción, pero `costo_reparacion_usd` se registra después de que ocurre la falla. Por eso es un atributo posterior al evento y una representación casi directa de la respuesta. La partición aleatoria también ignora ese orden temporal.
3. Además hay 37 registros duplicados al ignorar `id_registro`, que podrían quedar repartidos entre entrenamiento y prueba. La causa del AUC perfecto, sin embargo, es `costo_reparacion_usd`: su presencia equivale a `fallo = 1`. Completar los ausentes con cero no elimina la información; convierte la presencia/ausencia en una señal numérica todavía más explícita: costo positivo para los fallos y cero para los no fallos.

### Ejercicio 1 — Medir el modelo en las condiciones en que va a operar

La sospecha se confirma con una medición, no con un argumento. El modelo se entrenó viendo
`costo_reparacion_usd`, pero en el momento en que tiene que servir para algo —anticipar una
falla que **todavía no ocurrió**— esa columna llega vacía: el costo de la reparación se
registra después de la falla, no antes.

Corresponde implementar la evaluación del **mismo modelo ya ajustado**, sin reentrenarlo,
sobre el bloque de prueba en el que la columna llega ausente. El flujo de producción es el
mismo que el de entrenamiento: `preparar` completa la ausencia con el mismo cero, y
`transformar` aplica las mismas dos transformaciones ya ajustadas.

**El error previsto** es reentrenar el bosque sin la columna. Eso mide otra cosa —el modelo
honesto, que aparece dos celdas más abajo— y no la caída del modelo contaminado.

In [ ]:
def evaluar_en_produccion(modelo, bloque: pd.DataFrame, columna: str) -> float:
    """AUC del modelo cuando `columna` llega ausente, como en el momento de predecir.

    El modelo ya está ajustado y no se reentrena. Lo único que cambia es el bloque
    sobre el que se lo evalúa.

    Parámetros
    ----------
    modelo  : clasificador ya ajustado, con método `predict_proba`.
    bloque  : (n, 14) — las filas de prueba con todas sus columnas originales.
    columna : nombre de la columna que no está disponible al predecir.

    Devuelve
    --------
    float — AUC-ROC contra `y_pru`.
    """
    bloque_produccion = bloque.copy()
    bloque_produccion[columna] = np.nan
    probabilidades = modelo.predict_proba(
        transformar(preparar(bloque_produccion))
    )[:, 1]
    return float(roc_auc_score(y_pru, probabilidades))

In [ ]:
# VERIFICACIÓN — el mismo modelo, medido en las condiciones en que va a operar
auc_produccion = evaluar_en_produccion(bosque, pru, "costo_reparacion_usd")

assert 0.0 <= auc_produccion <= 1.0, f"un AUC no puede valer {auc_produccion}"
assert auc_produccion < auc_validacion, "en producción no puede ir mejor que en validación"

print(f"AUC en validación, evaluado como se entrenó : {auc_validacion:.4f}")
print(f"AUC en producción, con la columna ausente   : {auc_produccion:.4f}")
print(f"AUC perdida                                 : "
      f"{100 * (auc_validacion - auc_produccion) / auc_validacion:.1f} %")

Falta el término de comparación: **el modelo honesto**, el mismo flujo sin el atributo del
costo. Es el que parecía peor.

In [ ]:
# El modelo honesto, y la importancia de cada atributo en el contaminado. Celda dada.
A_hon, B_hon = ent[NUMERICAS].to_numpy(float), pru[NUMERICAS].to_numpy(float)
imp_hon = SimpleImputer(strategy="median").fit(A_hon)
esc_hon = StandardScaler().fit(imp_hon.transform(A_hon))
bosque_honesto = RandomForestClassifier(n_estimators=300, random_state=SEMILLA, n_jobs=-1).fit(
    esc_hon.transform(imp_hon.transform(A_hon)), y_ent)
auc_honesto = roc_auc_score(
    y_pru, bosque_honesto.predict_proba(esc_hon.transform(imp_hon.transform(B_hon)))[:, 1])

importancias = pd.Series(bosque.feature_importances_, index=ATRIBUTOS).sort_values()

print(f"con el atributo del costo, como se entrenó : {auc_validacion:.4f}")
print(f"el mismo modelo, en producción             : {auc_produccion:.4f}")
print(f"sin el atributo del costo, el honesto      : {auc_honesto:.4f}")
print(f"\nimportancia de costo_reparacion_usd en el modelo contaminado: "
      f"{importancias['costo_reparacion_usd']:.4f} de 1,0000")
print(f"la del atributo que le sigue: {importancias.iloc[-2]:.4f} ({importancias.index[-2]})")

fig, (izq, der) = plt.subplots(1, 2, figsize=(10.4, 3.8),
                               gridspec_kw={"width_ratios": [1, 1.25]})
etiquetas = ["En validación\n(como se entrenó)", "En producción\n(columna ausente)",
             "Modelo honesto\n(sin el atributo)"]
valores = [auc_validacion, auc_produccion, auc_honesto]
barras = izq.bar(etiquetas, valores, color=[NARANJA, NARANJA, OLIVA], width=0.62)
izq.bar_label(barras, fmt="%.4f", padding=3, fontsize=9)
izq.axhline(0.5, color=GRIS, ls="--", lw=1.2)
izq.text(-0.46, 0.52, "azar", fontsize=8.5, color=GRIS, ha="left")
izq.set_ylim(0, 1.14)
izq.set_ylabel("AUC-ROC")
izq.set_title("Tres mediciones del mismo flujo")
izq.tick_params(axis="x", labelsize=8)
izq.grid(False)

der.barh(importancias.index, importancias.to_numpy(),
         color=[NARANJA if c == "costo_reparacion_usd" else TURQUESA
                for c in importancias.index])
der.set_xlabel("Importancia en el modelo contaminado")
der.set_title("De dónde sale el AUC de 1,0000")
der.tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.show()

### Para analizar

1. `costo_reparacion_usd` está informado en $408$ filas, y en esas $408$ la máquina falló.
   Escribir qué relación exacta guarda entonces la columna con la respuesta, y por qué
   rellenarla con cero **la empeoró** en lugar de neutralizarla.
2. El modelo honesto supera al contaminado en las condiciones en que ambos van a operar.
   Un equipo que eligiera entre los dos mirando la primera medición se llevaría el peor.
   Escribir qué medición habría que haber mirado, y en qué momento del flujo estaba
   disponible.
3. La exposición señala que este atributo es **simultáneamente dos** de las cuatro formas de
   fuga. Nombrar las dos y decir cuál de las cuatro preguntas de revisión detecta cada una.

---

**Respuesta:**

1. La relación es exacta: `costo_reparacion_usd.notna()` si y solo si `fallo == 1` (408 de 408 fallos tienen costo y los 3829 no fallos no lo tienen). Al imputar cero se codificó esa relación como costo mayor que cero frente a costo igual a cero; por eso la señal quedó más fácil de explotar, no neutralizada.
2. Había que mirar una evaluación que reprodujera el momento real de predicción, con el costo todavía ausente. El mismo modelo contaminado cae de 1.0000 a 0.7306, mientras que el modelo honesto logra 0.7465. Esa medición debía formar parte del protocolo de validación antes de elegir el modelo y estaba disponible desde que se conocía qué variables existen al momento de predecir.
3. Es simultáneamente fuga por la respuesta y fuga temporal. La pregunta 2 —cuándo se registra cada columna— detecta que el costo revela el resultado; la pregunta 3 detecta que se usa información futura respecto del instante de predicción.

## 2. Dos fugas más, sobre el mismo conjunto

La de la sección 1 se ve en el AUC. Las dos de esta sección no: una **no deja rastro en la
salida**, y la otra **invierte la decisión** sobre un atributo sin que nada avise.

### 2.1 Duplicados repartidos entre particiones

La auditoría del conjunto ya encontró $37$ filas idénticas a otra fila, ignorando el
identificador: reingresos del mismo parte de mantenimiento, corrientes en sistemas donde un
operario puede cargarlo dos veces.

Para verlo con claridad conviene amplificarlo. La celda replica $800$ filas elegidas al azar
y compara dos procedimientos **que difieren en el orden de dos líneas**: quitar los
duplicados antes de partir, o después. La celda viene dada.

In [ ]:
clave = [c for c in df.columns if c != "id_registro"]

generador = np.random.default_rng(SEMILLA)
repetidas = generador.choice(len(df), size=800, replace=False)
con_reingresos = pd.concat([df, df.iloc[repetidas]], ignore_index=True)

print(f"conjunto con reingresos: {len(con_reingresos)} filas "
      f"({len(con_reingresos) - len(df)} repetidas)")

for etiqueta, quitar_antes in [("partir SIN quitar los duplicados", False),
                               ("quitar los duplicados y DESPUÉS partir", True)]:
    bloque = (con_reingresos.drop_duplicates(subset=clave).reset_index(drop=True)
              if quitar_antes else con_reingresos)
    yy = bloque["fallo"].to_numpy()
    Ad, Bd, ya, yb = train_test_split(bloque[NUMERICAS].to_numpy(float), yy,
                                      test_size=0.30, random_state=SEMILLA, stratify=yy)
    im = SimpleImputer(strategy="median").fit(Ad)
    mod = RandomForestClassifier(n_estimators=300, random_state=SEMILLA, n_jobs=-1).fit(
        im.transform(Ad), ya)
    auc = roc_auc_score(yb, mod.predict_proba(im.transform(Bd))[:, 1])
    print(f"  {etiqueta:<40} n = {len(bloque):>5}   AUC = {auc:.4f}")

### Para analizar

1. La diferencia es de trece puntos de AUC, y detrás no hay ningún modelo mejor: misma
   familia, mismos hiperparámetros, mismos datos. Escribir qué está haciendo el modelo
   cuando una fila está en ajuste y su copia exacta cae en prueba.
2. De las cuatro formas de fuga, esta es **la más fácil de corregir y la más fácil de pasar
   por alto**. Justificar las dos mitades de esa afirmación con lo que se acaba de medir.
3. Corresponde decidir dónde vive la corrección: ¿adentro o afuera del flujo que se ajusta
   sobre entrenamiento? Justificar a partir de qué estima esa operación.

**Respuesta:**

1. Cuando una observación queda en entrenamiento y su copia exacta cae en prueba, el bosque no generaliza: reconoce o memoriza un patrón que ya vio con la misma respuesta. La prueba deja de representar datos nuevos.
2. Es fácil de corregir porque basta eliminar duplicados reales antes de partir. Es fácil de pasar por alto porque `id_registro` hace que `df.duplicated()` considere distintas dos cargas idénticas y la métrica no avisa. En esta ejecución, no depurar produce AUC 0.8284 frente a 0.7318 luego de depurar: +0.0966 absoluto, o aproximadamente +13.2 % relativo, sin cambiar el modelo.
3. La corrección va fuera del pipeline y antes de la partición. Eliminar duplicados aplica una regla fija de igualdad y no estima parámetros estadísticos; hacerlo antes garantiza que una fila y su copia no puedan quedar en lados distintos.

### 2.2 La codificación que usa la respuesta

`id_maquina` tiene $180$ niveles para $4237$ filas: $23{,}5$ observaciones por nivel. Las
cuatro salidas disponibles, con su costo en columnas:

| Salida | Columnas nuevas | Qué supone |
|---|---|---|
| One-hot | $179$ | Que hay datos para estimar 179 coeficientes con 23,5 observaciones cada uno |
| Por frecuencia | $1$ | Que la frecuencia del nivel es informativa |
| Agrupar los niveles raros y one-hot | $138$ | Que los niveles con menos de 20 observaciones son intercambiables |
| **Por la respuesta** | $1$ | Que cada nivel tiene observaciones suficientes para estimar su promedio |

La cuarta es la más eficiente y la más peligrosa: resume en **una** columna lo que one-hot
necesita 179 para expresar, y lo hace **usando la respuesta**, que es justamente lo que el
modelo tiene que predecir.

La celda siguiente entrega el andamiaje de medición —una validación cruzada de 5
particiones que recibe una función constructora de la columna— y dos de los cuatro
procedimientos: no usar la columna, y calcular el promedio sobre el conjunto completo.

In [ ]:
# Andamiaje de medición. Celda dada.
X_numerica = df[NUMERICAS].to_numpy(float)
maquina = df["id_maquina"].to_numpy()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

print(f"niveles de id_maquina: {df.id_maquina.nunique()} · "
      f"observaciones por nivel: {len(df) / df.id_maquina.nunique():.1f}")


def auc_cv(construir_columna, etiqueta: str) -> float:
    """AUC-ROC promedio de 5 particiones, agregando la columna que devuelve `construir_columna`.

    `construir_columna(itr, iva)` recibe los índices de ajuste y de validación del
    fold y devuelve el par (columna de ajuste, columna de validación).
    """
    aucs = []
    for itr, iva in cv.split(X_numerica, y):
        col_tr, col_va = construir_columna(itr, iva)
        Acv = np.column_stack([X_numerica[itr], col_tr])
        Bcv = np.column_stack([X_numerica[iva], col_va])
        im = SimpleImputer(strategy="median").fit(Acv)
        es = StandardScaler().fit(im.transform(Acv))
        mod = LogisticRegression(max_iter=5000).fit(es.transform(im.transform(Acv)), y[itr])
        aucs.append(roc_auc_score(y[iva], mod.predict_proba(
            es.transform(im.transform(Bcv)))[:, 1]))
    print(f"  {etiqueta:<52} AUC-VC = {np.mean(aucs):.4f} (± {np.std(aucs):.4f})")
    return float(np.mean(aucs))


promedio_global = pd.Series(y).groupby(maquina).mean()


def columna_sin(itr, iva):
    """Una columna de ceros: equivale a no usar el atributo."""
    return np.zeros(len(itr)), np.zeros(len(iva))


def columna_con_fuga(itr, iva):
    """El promedio de la respuesta por nivel, calculado sobre TODO el conjunto."""
    col = pd.Series(maquina).map(promedio_global).to_numpy(float)
    return col[itr], col[iva]

### Ejercicio 2 — La misma codificación, calculada fuera de fold y suavizada

Corresponde escribir el procedimiento correcto, que tiene dos partes.

**Primera parte: calcular la columna dentro de cada fold.** El promedio de cada nivel se
estima usando **solo** las observaciones de ajuste del fold, y se aplica a las de validación.

**Segunda parte: suavizar.** Con $23{,}5$ observaciones por nivel, el promedio de un nivel es
una estimación ruidosa. Se lo mezcla con el promedio global mediante un peso $\kappa$ que
hace de cantidad de observaciones ficticias: un nivel con muchas observaciones conserva su
promedio, y uno con pocas se acerca al global.

El pseudocódigo es el de la exposición:

```text
Entrada: categórica c, respuesta y, índices de ajuste A, índices de validación V, peso κ
Salida:  el par (columna de ajuste, columna de validación)

1.  prior ← media de y sobre A
2.  para cada nivel ℓ presente en A:
3.      n_ℓ ← cantidad de observaciones de nivel ℓ en A
4.      m_ℓ ← media de y en esas observaciones
5.      v[ℓ] ← (m_ℓ·n_ℓ + prior·κ) / (n_ℓ + κ)
6.  asignar v[ℓ] a las observaciones de A y de V según su nivel
7.  los niveles que no aparecen en A reciben prior
8.  devolver las dos columnas

Costo: O(n) por fold
```

**Dos advertencias.**

- **Choque de símbolos.** Acá $\kappa$ es el peso del suavizado. El $k$ de la sección 4 es la
  cantidad de valores observados de una columna. En el código son `k_suavizado` y `k`, para
  que no se confundan.
- **El error previsto** es calcular `prior` sobre `y` completo en lugar de sobre `y[itr]`.
  Con $\kappa = 0$ el suavizado no interviene y el error queda invisible; con $\kappa > 0$
  filtra la respuesta de validación por la puerta de atrás. **El paso 1 dice `sobre A`.**

In [ ]:
def columna_fuera_de_fold(itr: np.ndarray, iva: np.ndarray,
                          k_suavizado: float = 0.0) -> tuple[np.ndarray, np.ndarray]:
    """Codifica `maquina` por la respuesta, estimando solo sobre las filas de ajuste.

    Parámetros
    ----------
    itr         : (n_tr,) — índices de ajuste del fold.
    iva         : (n_va,) — índices de validación del fold.
    k_suavizado : peso del promedio global, en observaciones ficticias. Con 0 no
                  hay suavizado y cada nivel conserva su promedio crudo.

    Devuelve
    --------
    (col_tr, col_va) — de formas (n_tr,) y (n_va,). Los niveles ausentes en el
    ajuste reciben el promedio global del ajuste.
    """
    prior = float(y[itr].mean())
    ajuste = pd.DataFrame({"maquina": maquina[itr], "respuesta": y[itr]})
    resumen = ajuste.groupby("maquina")["respuesta"].agg(["mean", "count"])
    valores = (resumen["mean"] * resumen["count"] + prior * k_suavizado) / (
        resumen["count"] + k_suavizado
    )
    col_tr = pd.Series(maquina[itr]).map(valores).fillna(prior).to_numpy(float)
    col_va = pd.Series(maquina[iva]).map(valores).fillna(prior).to_numpy(float)
    return col_tr, col_va

La verificación no compara contra una cifra: comprueba **la propiedad que define al
procedimiento**. Si la columna de validación no usa la respuesta de validación, entonces
alterar esa respuesta no puede cambiarla.

In [ ]:
# VERIFICACIÓN — la columna de validación no puede depender de la respuesta de validación
itr, iva = next(iter(cv.split(X_numerica, y)))
col_tr, col_va = columna_fuera_de_fold(itr, iva, 0.0)

assert col_tr.shape == (len(itr),), f"forma incorrecta en ajuste: {col_tr.shape}"
assert col_va.shape == (len(iva),), f"forma incorrecta en validación: {col_va.shape}"

_, col_va_suave = columna_fuera_de_fold(itr, iva, 20.0)

y_intacto = y
try:
    y = y_intacto.copy()
    y[iva] = 1 - y[iva]                       # se invierte la respuesta de validación
    _, col_va_alterada = columna_fuera_de_fold(itr, iva, 20.0)
finally:
    y = y_intacto

assert np.allclose(col_va_suave, col_va_alterada), (
    "invertir y[iva] cambió la columna de validación: el procedimiento tiene fuga")
print("Invertir la respuesta de validación no movió la columna de validación.")

# Un nivel que no aparece en el ajuste tiene que recibir el promedio global del ajuste
nivel_aislado = pd.Series(maquina).value_counts().index[-1]
solo_validacion = np.where(maquina == nivel_aislado)[0]
resto = np.setdiff1d(np.arange(len(y)), solo_validacion)
_, col_no_vista = columna_fuera_de_fold(resto, solo_validacion, 0.0)

assert np.allclose(col_no_vista, y[resto].mean()), (
    "un nivel ausente del ajuste no recibió el promedio global")
print(f"El nivel {nivel_aislado}, ausente del ajuste, recibió el promedio global "
      f"{y[resto].mean():.4f} en sus {len(solo_validacion)} filas.")

In [ ]:
# Los cuatro procedimientos, sobre la misma validación cruzada. Celda dada.
a_sin = auc_cv(columna_sin, "sin la columna")
a_fuga = auc_cv(columna_con_fuga, "promedio sobre TODO el conjunto (con fuga)")
a_oof = auc_cv(lambda t, v: columna_fuera_de_fold(t, v, 0.0),
               "promedio dentro de cada fold")
a_suave = auc_cv(lambda t, v: columna_fuera_de_fold(t, v, 20.0),
                 "ídem, con suavizado kappa = 20")

print(f"\n  la fuga reporta {100 * (a_fuga - a_sin) / a_sin:+.2f} % sobre no usar la columna")
print(f"  el procedimiento correcto reporta {100 * (a_oof - a_sin) / a_sin:+.2f} %")
print(f"  con suavizado, {100 * (a_suave - a_sin) / a_sin:+.2f} %")

### Para analizar

1. El procedimiento contaminado reporta una **mejora**; el correcto, una **pérdida**.
   Escribir qué decisión concreta sobre `id_maquina` toma cada uno, y qué se lleva a
   producción quien mira el primero.
2. Esta fuga no infla el número tanto como la de la sección 1 —son cuatro puntos y medio
   porcentuales, no veintisiete— y sin embargo es **peor** en un sentido preciso. Decir en
   cuál.
3. El suavizado recupera dos décimas respecto del procedimiento crudo fuera de fold.
   Escribir en una línea qué **no** logra el suavizado, retomando el corolario de la
   exposición: hacer bien una transformación no la vuelve buena.

---

**Respuesta:**

1. El procedimiento contaminado elegiría incluir `id_maquina`, porque aparenta subir el AUC de 0.7783 a 0.8154. El procedimiento correcto muestra lo contrario: baja a 0.7414, o a 0.7435 con suavizado. Quien mira el primero lleva a producción un atributo sin aporte generalizable y empeora el modelo.
2. Es peor porque no solo sobreestima el desempeño: invierte una decisión de selección de atributos. La fuga queda escondida dentro de una transformación aparentemente válida y hace elegir de manera sistemática el modelo que rendirá peor con datos nuevos.
3. El suavizado reduce la varianza de los promedios de niveles con pocos casos, pero no crea señal fuera de muestra ni vuelve útil a `id_maquina`; una transformación correctamente calculada puede seguir siendo una mala decisión predictiva.

## 3. La estructura que lo impide

Las tres secciones anteriores son una lista de cosas que hay que recordar, y ese es
exactamente el problema: **el orden correcto no es verificable leyendo el código.** Un
`fit_transform` sobre el conjunto completo y otro sobre el conjunto de ajuste se ven
idénticos, y la diferencia puede estar a veinte líneas de distancia.

La respuesta es estructural. Pero antes de armarla conviene separar **lo que sí puede ir
antes de partir**, y por un motivo preciso: son las transformaciones que **no estiman nada**.

| Operación | Qué estima | Dónde va |
|---|---|---|
| Convertir los centinelas `999` y `-1` a faltante | Nada: es una regla fija | Antes de partir |
| Normalizar el texto de `modelo_equipo` | Nada: es una regla fija | Antes de partir |
| Eliminar las filas duplicadas | Nada | **Antes de partir, y es obligatorio** |
| Imputación, escalado, one-hot | La mediana, el desvío, los niveles | **Dentro del pipeline** |

La celda siguiente aplica las tres primeras. Viene dada.

In [ ]:
# Las tres reglas fijas, que no estiman nada y por eso pueden ir antes de partir.
limpio = df.copy()
limpio["temperatura_c"] = limpio["temperatura_c"].replace(999.0, np.nan)
limpio["presion_bar"] = limpio["presion_bar"].replace(-1.0, np.nan)
limpio["modelo_equipo"] = (limpio["modelo_equipo"].str.upper()
                           .str.replace("-", "", regex=False))
limpio = limpio.drop_duplicates(subset=clave).reset_index(drop=True)

X_limpio = limpio[NUMERICAS + CATEGORICAS]
y_limpio = limpio["fallo"].to_numpy()

print(f"filas: {len(df)} -> {len(limpio)}  (se quitaron los reingresos)")
print(f"centinelas convertidos a faltante: "
      f"temperatura_c {int(limpio.temperatura_c.isna().sum())} · "
      f"presion_bar {int(limpio.presion_bar.isna().sum())}")
print(f"niveles de modelo_equipo: {df.modelo_equipo.nunique()} -> "
      f"{limpio.modelo_equipo.nunique()}  (la codificación era inconsistente)")

### Ejercicio 3 — El flujo como `Pipeline` con `ColumnTransformer`

Corresponde construir el flujo completo como un solo objeto, con **dos ramas** sobre grupos
de columnas distintos:

| Rama | Columnas | Pasos |
|---|---|---|
| Numérica | `NUMERICAS` | imputar por la **mediana**, después **escalado robusto** |
| Categórica | `CATEGORICAS` | imputar con la **categoría faltante**, después **one-hot** |

Tres decisiones y su justificación, todas de la exposición:

- **Escalado robusto y no estandarización.** El conjunto tenía centinelas, y aunque acá ya
  se limpiaron, en un flujo con decenas de columnas alguno se escapa. Restar la mediana y
  dividir por el rango intercuartílico no se desarma con un valor extremo.
- **Categoría faltante y no la moda.** Un nivel explícito no imputa nada, no puede sesgar la
  distribución de los niveles reales y preserva la información de la ausencia. Es la opción
  por omisión razonable para categóricas. En este conjunto las categóricas no tienen
  faltantes, de modo que el paso no cambia ningún valor: **está por estructura, no por
  necesidad**, que es justamente el punto de la sección.
- **One-hot que tolere niveles no vistos.** Un nivel que aparece en validación y no estaba
  en ajuste no puede hacer fallar el flujo.

El modelo final es una regresión logística, que es sensible a la escala y por lo tanto sirve
para que el escalado tenga consecuencias observables.

In [ ]:
def construir_pipeline() -> Pipeline:
    """Arma el flujo completo: dos ramas de preparación y el modelo, en un solo objeto.

    Devuelve
    --------
    Pipeline sin ajustar, con dos pasos: "preparar" (un ColumnTransformer con las
    ramas "num" y "cat") y "modelo" (una LogisticRegression con max_iter=5000).

    La rama "num" se llama así y se aplica a NUMERICAS; la rama "cat", a
    CATEGORICAS. Los nombres importan: la verificación los usa para recuperar los
    estadísticos que el pipeline aprendió.
    """
    rama_num = Pipeline([
        ("imputar", SimpleImputer(strategy="median")),
        ("escalar", RobustScaler()),
    ])
    rama_cat = Pipeline([
        ("imputar", SimpleImputer(strategy="constant", fill_value="Faltante")),
        ("codificar", OneHotEncoder(handle_unknown="ignore", drop="first")),
    ])
    preparar = ColumnTransformer([
        ("num", rama_num, NUMERICAS),
        ("cat", rama_cat, CATEGORICAS),
    ])
    return Pipeline([
        ("preparar", preparar),
        ("modelo", LogisticRegression(max_iter=5000)),
    ])

La verificación es el punto de toda la sección, y no es una cifra: es una **propiedad
estructural**. Si cada rama se ajusta solo con los datos que le tocan, entonces dos
particiones distintas tienen que producir **estadísticos distintos**, y ninguna de las dos
puede haber aprendido el estadístico del conjunto completo.

In [ ]:
# VERIFICACIÓN — cada fold aprende sus propios estadísticos, y ninguno aprende los globales
def medianas_aprendidas(indices: np.ndarray) -> np.ndarray:
    """Las medianas que el imputador de la rama numérica estimó al ajustar."""
    ajustado = construir_pipeline().fit(X_limpio.iloc[indices], y_limpio[indices])
    rama = ajustado.named_steps["preparar"].named_transformers_["num"]
    return rama.named_steps["imputar"].statistics_


particiones = list(cv.split(X_limpio, y_limpio))
med_1 = medianas_aprendidas(particiones[0][0])
med_2 = medianas_aprendidas(particiones[1][0])
med_global = X_limpio[NUMERICAS].median().to_numpy()

assert not np.allclose(med_1, med_global), "el fold 1 aprendió los estadísticos globales"
assert not np.allclose(med_2, med_global), "el fold 2 aprendió los estadísticos globales"
assert not np.allclose(med_1, med_2), "los dos folds aprendieron lo mismo"

comparacion = pd.DataFrame({"global": med_global, "fold 1": med_1, "fold 2": med_2},
                           index=NUMERICAS)
print(comparacion.round(4).to_string())
print(f"\nmáxima diferencia entre los dos folds: {np.abs(med_1 - med_2).max():.4f}")
print("Ningún fold vio la mediana del conjunto completo: la fuga no es escribible.")

In [ ]:
# Los dos protocolos, medidos. Celda dada.
pipe = construir_pipeline()
auc_pipe = cross_val_score(pipe, X_limpio, y_limpio, cv=cv, scoring="roc_auc")

# El protocolo mal armado: preparar TODO el conjunto y recién después validar.
preparacion_sola = construir_pipeline().named_steps["preparar"]
X_preparado = preparacion_sola.fit_transform(X_limpio)
auc_mal = cross_val_score(LogisticRegression(max_iter=5000), X_preparado, y_limpio,
                          cv=cv, scoring="roc_auc")

print(f"la validación envuelve al pipeline    AUC-VC = {auc_pipe.mean():.4f} "
      f"(± {auc_pipe.std():.4f})")
print(f"preparar todo y después validar       AUC-VC = {auc_mal.mean():.4f} "
      f"(± {auc_mal.std():.4f})")
print(f"brecha: {auc_mal.mean() - auc_pipe.mean():+.4f}")

### Para analizar

1. La brecha entre los dos protocolos es de **una diezmilésima de AUC**. Escribir por qué,
   con estas transformaciones concretas, era esperable que fuera chica —y decir qué estiman
   la imputación por la mediana y el escalado robusto que el conjunto de validación pudiera
   revelar.
2. Si el pipeline no mejora el número, corresponde justificar por qué se usa igual. La
   respuesta está en la verificación de dos celdas atrás, no en esta.
3. Las diferencias chicas se reportan como chicas: inflar esta sería el mismo error que el
   laboratorio entero denuncia. Escribir qué conclusión **no** autoriza esta brecha.

**Respuesta:**

1. Era esperable una brecha mínima porque hay miles de filas y pocos faltantes, de modo que mediana e IQR cambian poco entre folds. Preparar todo antes de validar deja que la validación influya en las medianas usadas para imputar y en la mediana y el rango intercuartílico usados por `RobustScaler`; esa información es real aunque aquí apenas mueva el AUC.
2. Se usa por la garantía estructural: cada fold ajusta sus propios estadísticos solo con sus filas de entrenamiento. Las medianas distintas entre folds y respecto de la global verifican que la validación no fue observada, y la misma estructura protege cualquier evaluación futura.
3. La brecha cercana a cero no autoriza a concluir que preparar antes de validar sea correcto, que la fuga no exista o que un pipeline mejore el desempeño. Solo dice que, en este conjunto y con estas transformaciones, el efecto medido fue diminuto.

### El remate: la fuga que sí se medía

La brecha anterior es chica porque las transformaciones de las dos ramas apenas filtran. La
codificación por la respuesta de la sección 2.2 **sí filtraba**, y mucho: reportaba una
mejora del $4{,}8\ \%$ que en realidad era una pérdida.

La celda siguiente mide tres protocolos sobre el conjunto limpio: sin `id_maquina`, con
`id_maquina` codificado **dentro** del pipeline, y con `id_maquina` codificado **antes** de
que la validación empiece. Viene dada.

In [ ]:
# Tres protocolos, la misma validación cruzada. Celda dada.
from sklearn.preprocessing import TargetEncoder    # requiere scikit-learn 1.3 o posterior

X_con_maquina = limpio[NUMERICAS + CATEGORICAS + ["id_maquina"]]

# (a) sin el atributo
a_a = cross_val_score(construir_pipeline(), X_limpio, y_limpio,
                      cv=cv, scoring="roc_auc").mean()

# (b) codificado por la respuesta DENTRO del pipeline: TargetEncoder estima el
#     promedio por nivel con validación interna, y solo con los datos del paso.
rama_num_b = Pipeline([("imputar", SimpleImputer(strategy="median")),
                       ("escalar", RobustScaler())])
rama_cat_b = Pipeline([("imputar", SimpleImputer(strategy="constant", fill_value="Faltante")),
                       ("codificar", OneHotEncoder(handle_unknown="ignore", drop="first"))])
pipe_dentro = Pipeline([
    ("preparar", ColumnTransformer([
        ("num", rama_num_b, NUMERICAS),
        ("cat", rama_cat_b, CATEGORICAS),
        ("maq", TargetEncoder(target_type="binary", random_state=SEMILLA), ["id_maquina"]),
    ])),
    ("modelo", LogisticRegression(max_iter=5000)),
])
a_b = cross_val_score(pipe_dentro, X_con_maquina, y_limpio, cv=cv, scoring="roc_auc").mean()

# (c) codificado ANTES de que la validación empiece
mapa_limpio = pd.Series(y_limpio).groupby(limpio["id_maquina"].to_numpy()).mean()
X_antes = X_limpio.copy()
X_antes["maq_codificada"] = limpio["id_maquina"].map(mapa_limpio).to_numpy(float)
pipe_antes = Pipeline([
    ("preparar", ColumnTransformer([
        ("num", rama_num_b, NUMERICAS + ["maq_codificada"]),
        ("cat", rama_cat_b, CATEGORICAS),
    ])),
    ("modelo", LogisticRegression(max_iter=5000)),
])
a_c = cross_val_score(pipe_antes, X_antes, y_limpio, cv=cv, scoring="roc_auc").mean()

print(f"(a) sin id_maquina                              AUC-VC = {a_a:.4f}")
print(f"(b) id_maquina codificado DENTRO del pipeline   AUC-VC = {a_b:.4f}   "
      f"({100 * (a_b - a_a) / a_a:+.2f} % sobre (a))")
print(f"(c) id_maquina codificado ANTES de validar      AUC-VC = {a_c:.4f}   "
      f"({100 * (a_c - a_a) / a_a:+.2f} % sobre (a))")

### Para analizar

1. El protocolo (c) reproduce la ilusión de la sección 2.2 sobre el conjunto limpio. El (b)
   la hace desaparecer. Escribir qué línea de código distingue a uno del otro, y por qué
   dentro del pipeline **no hay manera de escribir** el (c).
2. Los protocolos (a) y (b) coinciden hasta la cuarta cifra. Decir qué afirmación sobre
   `id_maquina` autoriza esa coincidencia, y contrastarla con lo que afirmaría quien mirara
   solo el (c).

---

**Respuesta:**

1. En (c), `mapa_limpio = pd.Series(y_limpio).groupby(...).mean()` se ejecuta sobre toda la respuesta antes de `cross_val_score`; por eso cada fila de validación influye en su propia codificación. En (b), `TargetEncoder` está dentro del pipeline: la validación cruzada clona el objeto y llama a `fit` únicamente con `X[itr]` e `y[itr]`. Desde dentro de ese ajuste no existe acceso a `y[iva]`, así que la variante contaminada no se puede expresar.
2. Como (a) y (b) dan 0.7842, `id_maquina` no aporta información predictiva fuera de muestra bajo este protocolo. Mirar solo (c), con AUC 0.8203 (+4.61 %), llevaría a afirmar falsamente que la identidad de la máquina mejora el modelo.

## 4. La ausencia como dato

Preparar los datos empieza por los faltantes, y el error más común no es elegir mal la
imputación: es no preguntarse **por qué falta lo que falta**. La respuesta determina qué
estrategias son válidas y cuáles introducen un sesgo.

Con $M$ el indicador de ausencia —$M_{ij} = 1$ cuando el valor correspondiente falta— y el
conjunto partido en la porción observada $X_{\text{obs}}$ y la ausente $X_{\text{mis}}$, los
tres mecanismos son condiciones sobre la distribución de $M$:

$$
\textbf{MCAR:}\quad P(M \mid X_{\text{obs}}, X_{\text{mis}}) = P(M)
$$

$$
\textbf{MAR:}\quad P(M \mid X_{\text{obs}}, X_{\text{mis}}) = P(M \mid X_{\text{obs}})
$$

$$
\textbf{MNAR:}\quad P(M \mid X_{\text{obs}}, X_{\text{mis}}) \ \text{depende de}\ X_{\text{mis}}
$$

Los faltantes de este conjunto **fueron inyectados con los tres mecanismos**, uno por
columna, de modo que el diagnóstico se puede contrastar contra la verdad. Cuál es cuál no se
revela hasta el final de la sección.

### Ejercicio 4 — El diagnóstico

El procedimiento es el mismo para cada columna con ausencias: se construye la indicadora de
ausencia y se pregunta si está asociada con las demás columnas observadas. El pseudocódigo
es el de la exposición:

```text
Entrada: columna c con faltantes, una categórica z, las columnas numéricas, nivel alfa
Salida:  el p del contraste contra z, y la lista de numéricas asociadas

1.  M ← 1{c está ausente}
2.  tabla de contingencia M × z, contraste chi-cuadrado -> chi2, p_z
3.  para cada numérica w distinta de c:
4.      comparar la media de w entre M=0 y M=1, con un contraste de medias
5.      si el p resulta menor que alfa: registrar (w, media con c presente,
6.                                                    media con c ausente, p)
7.  devolver (chi2, p_z, la lista de registradas)

Si ninguna asociación resulta significativa: no se puede rechazar MCAR.
Si alguna resulta significativa: MCAR queda descartado, y es MAR o MNAR —
                                 pero eso NO se puede distinguir.

Costo: O(np) contrastes sobre p columnas
```

**Dos detalles de implementación.** Las varianzas de los dos grupos no tienen por qué ser
iguales, de modo que el contraste de medias va **sin suponer varianzas iguales**. Y cada
columna numérica puede tener sus propios faltantes: hay que descartarlos antes de comparar,
o el promedio sale `nan`.

**El error previsto** es construir $M$ con `== np.nan`, que siempre da falso. La ausencia se
consulta con el método propio de la biblioteca.

In [ ]:
def diagnosticar(datos: pd.DataFrame, col: str, categorica: str = "turno",
                 numericas: list[str] | None = None,
                 alfa: float = 0.01) -> tuple[float, float, list[tuple]]:
    """Diagnostica el mecanismo de ausencia de `col` contra las columnas observadas.

    Parámetros
    ----------
    datos      : el conjunto completo, con los faltantes sin imputar.
    col        : nombre de la columna cuya ausencia se diagnostica.
    categorica : columna categórica contra la que se hace el contraste chi-cuadrado.
    numericas  : columnas numéricas a contrastar; `col` se excluye sola.
    alfa       : nivel del contraste.

    Devuelve
    --------
    (chi2, p_categorica, asociadas) donde `asociadas` es la lista de tuplas
    (nombre, media con `col` presente, media con `col` ausente, p) de las
    numéricas con p < alfa, en el orden de `numericas`.
    """
    numericas = NUMERICAS if numericas is None else numericas
    falta = datos[col].isna()
    tabla = pd.crosstab(falta, datos[categorica])
    chi2, p_categorica, _, _ = stats.chi2_contingency(tabla)

    asociadas = []
    for nombre in numericas:
        if nombre == col:
            continue
        presente = datos.loc[~falta, nombre].dropna()
        ausente = datos.loc[falta, nombre].dropna()
        _, p = stats.ttest_ind(presente, ausente, equal_var=False)
        if p < alfa:
            asociadas.append((nombre, presente.mean(), ausente.mean(), p))

    return float(chi2), float(p_categorica), asociadas

In [ ]:
# VERIFICACIÓN — el diagnóstico corre sobre las tres columnas con faltantes
CON_FALTANTES = ["consumo_kwh", "vibracion_mm_s", "temperatura_c"]
diagnostico = {}

for col in CON_FALTANTES:
    chi2, p_turno, asociadas = diagnosticar(df, col)
    diagnostico[col] = (chi2, p_turno, asociadas)
    falta = df[col].isna()
    tasas = falta.groupby(df["turno"]).mean()

    print(f"--- {col}: {int(falta.sum())} faltantes ({100 * falta.mean():.2f} %)")
    print("    tasa de ausencia por turno: "
          + " · ".join(f"{t} {100 * v:.2f} %" for t, v in tasas.items()))
    print(f"    chi-cuadrado contra turno: {chi2:.2f}, p = {p_turno:.3e}   "
          f"{'ASOCIACIÓN' if p_turno < 0.01 else 'sin asociación'}")
    for nombre, m_pres, m_aus, p in asociadas:
        print(f"    {nombre:>18}: presente {m_pres:9.3f} · faltante {m_aus:9.3f}  "
              f"p = {p:.2e}  ASOCIACIÓN")
    if not asociadas:
        print("    ninguna numérica asociada")
    hay_asociacion = p_turno < 0.01 or bool(asociadas)
    print("    VEREDICTO: " + ("MCAR descartado; es MAR o MNAR, y no se puede distinguir"
                               if hay_asociacion else "no se puede rechazar MCAR"))

# La primera columna no muestra asociación con nada; la segunda, con el turno.
assert diagnostico["consumo_kwh"][1] > 0.01 and not diagnostico["consumo_kwh"][2]
assert diagnostico["vibracion_mm_s"][1] < 1e-100
# La tercera parece MCAR contra el turno, y no lo es.
assert diagnostico["temperatura_c"][1] > 0.01
assert len(diagnostico["temperatura_c"][2]) == 2
print("\nLas tres verificaciones pasan.")

In [ ]:
# La misma evidencia, en cuatro paneles. Celda dada.
fig, ejes = plt.subplots(1, 4, figsize=(13.4, 3.5))
colores = [TURQUESA, NARANJA, VIOLETA]

for eje, col, color in zip(ejes, CON_FALTANTES, colores):
    falta = df[col].isna()
    tasas = falta.groupby(df["turno"]).mean() * 100
    eje.bar(tasas.index, tasas.to_numpy(), color=color, width=0.6)
    eje.axhline(100 * falta.mean(), color=GRIS, ls="--", lw=1.2)
    eje.set_title(f"{col}\ncontra el turno", fontsize=9.5)
    eje.set_ylabel("Ausencia (%)" if col == CON_FALTANTES[0] else "")
    eje.set_ylim(0, 38)
    eje.tick_params(axis="x", labelsize=8.5)
    for i, v in enumerate(tasas.to_numpy()):
        eje.text(i, v + 0.8, f"{v:.1f} %".replace(".", ","), ha="center", fontsize=8.5)
    eje.grid(False)

# El cuarto panel: el rastro que la ausencia de temperatura deja en una observada.
falta_temp = df["temperatura_c"].isna()
presente = df.loc[~falta_temp, "consumo_kwh"].dropna()
ausente = df.loc[falta_temp, "consumo_kwh"].dropna()
ejes[3].hist(presente, bins=40, density=True, color=GRIS, alpha=0.65,
             label="temperatura presente")
ejes[3].hist(ausente, bins=22, density=True, histtype="step", color=VIOLETA, lw=2.2,
             label="temperatura ausente")
ejes[3].axvline(presente.mean(), color=GRIS, ls="--", lw=1.4)
ejes[3].axvline(ausente.mean(), color=VIOLETA, ls="--", lw=1.4)
ejes[3].set_title("temperatura_c\nel rastro en consumo_kwh", fontsize=9.5)
ejes[3].set_xlabel("consumo_kwh")
ejes[3].set_ylabel("Densidad")
ejes[3].legend(fontsize=7.5, loc="upper right")
plt.tight_layout()
plt.show()

t_obs = df.loc[df.temperatura_c < 900, "temperatura_c"].dropna()
print(f"temperatura observada: máximo {t_obs.max():.2f} °C, "
      f"percentil 99 {t_obs.quantile(0.99):.2f} °C")

### DECIDE

Los dos primeros casos son concluyentes: uno no muestra asociación con nada, y el otro tiene
una asociación abrumadora con una variable **registrada**. El tercero es el interesante.

1. Asignar un mecanismo a cada una de las tres columnas, con la evidencia que lo sostiene.
2. Sobre `temperatura_c`: contra el turno parece MCAR, y contra las numéricas no lo es.
   Escribir qué queda descartado y **qué no se puede decidir**. La hipótesis de que el sensor
   se satura por encima de cierto valor y la hipótesis de que la ausencia depende del consumo
   explican la misma tabla: decir por qué ningún contraste sobre estos datos las separa.
3. Corresponde decidir qué se hace con `temperatura_c` en el flujo de producción, sabiendo
   que el mecanismo quedó indeterminado. **La respuesta tiene dos partes**: la estrategia, y
   qué se reporta junto con el resultado.
4. El máximo observado y el percentil 99 de la temperatura están impresos arriba. Decir qué
   verificación **por fuera de los datos** confirmaría o descartaría la saturación del
   sensor.

**Respuesta:**

1. `consumo_kwh` es compatible con MCAR: ni el turno (p = 0.4683) ni las demás numéricas muestran asociación con su ausencia. `vibracion_mm_s` es MAR: la ausencia depende fuertemente del turno observado (33.14 % de noche frente a 2–3 % en los otros turnos; p = 1.66e-166). `temperatura_c` corresponde al caso MNAR según la verdad de generación —se inyectó un mecanismo de cada tipo—, aunque los datos observados por sí solos solo permiten clasificarla como MAR o MNAR: su ausencia se asocia con vibración (p = 6.76e-4) y consumo (p = 1.25e-12).
2. Para `temperatura_c` se descarta MCAR, pero no se puede decidir entre MAR y MNAR. Cuando la temperatura falta no observamos su valor, así que no podemos contrastar directamente si la probabilidad de ausencia depende de la propia temperatura. Tanto una saturación a temperaturas altas como una dependencia del consumo observado producen las asociaciones encontradas.
3. En producción conviene imputar de forma robusta dentro del pipeline y agregar un indicador de ausencia, conservando la señal de que el sensor no informó. Junto con el resultado debe declararse que el mecanismo MAR/MNAR es indeterminado y reportarse un análisis de sensibilidad bajo hipótesis alternativas; la imputación no resuelve esa incertidumbre.
4. Hay que consultar la ficha técnica, el rango operativo y los registros de calibración/telemetría del sensor, o hacer una prueba controlada cerca y por encima de 99 °C. Eso permitiría comprobar si el equipo satura, corta o deja de registrar a partir de un umbral.

### 4.1 Qué le hace la imputación a los datos

La estrategia de uso corriente reemplaza cada faltante por la media, la mediana o la moda de
la columna. Es un modelo —su único parámetro se estima a partir de datos— y su efecto sobre
la distribución **se puede calcular exactamente**.

Sea una columna con $n$ valores, de los cuales $k$ están observados con media
$\bar x_{\text{obs}}$ y varianza muestral $s^2_{\text{obs}}$, y $m = n - k$ ausentes, todos
reemplazados por $\bar x_{\text{obs}}$. La media no cambia, y los $m$ valores imputados
valen exactamente esa media, de modo que aportan **cero** a la suma de cuadrados. El
numerador queda igual y el denominador pasa de $k-1$ a $n-1$:

$$
s^2_{\text{imp}} = \frac{(k-1)\,s^2_{\text{obs}}}{n-1}
= \underbrace{\frac{k-1}{n-1}}_{<\,1}\; s^2_{\text{obs}}
$$

### Ejercicio 5 — La cuenta, contra la medición

Corresponde medir las dos varianzas y calcular el factor, para contrastar la predicción de la
fórmula contra lo que efectivamente pasa. **Son tres líneas.**

**El error previsto** es el denominador. La varianza muestral lleva $\text{ddof}=1$; con el
denominador de la biblioteca por omisión —que es $n$— la identidad no cierra y el resultado
no se distingue de un error de implementación.

In [ ]:
def varianza_tras_imputar(x: np.ndarray) -> tuple[float, float, float]:
    """Varianza antes y después de imputar la media, y el factor que las relaciona.

    Parámetros
    ----------
    x : (n,) — la columna, con `np.nan` en los valores ausentes.

    Devuelve
    --------
    (s2_obs, s2_imp, factor) donde
      s2_obs : varianza muestral (ddof=1) sobre los k valores observados.
      s2_imp : varianza muestral (ddof=1) de la columna completa, con los m
               ausentes reemplazados por la media de los observados.
      factor : (k-1)/(n-1), la predicción de la fórmula.
    """
    observados = x[~np.isnan(x)]
    imputada = np.where(np.isnan(x), observados.mean(), x)
    factor = (len(observados) - 1) / (len(x) - 1)
    return float(np.var(observados, ddof=1)), float(np.var(imputada, ddof=1)), float(factor)

In [ ]:
# VERIFICACIÓN — la fórmula contra la medición, sobre consumo_kwh
columna = df["consumo_kwh"].to_numpy(float)
n = len(columna)
k = int(np.sum(~np.isnan(columna)))
s2_obs, s2_imp, factor = varianza_tras_imputar(columna)

assert np.isclose(factor, (k - 1) / (n - 1)), "el factor no es (k-1)/(n-1)"
assert np.allclose(s2_obs * factor, s2_imp), (
    f"la identidad no cierra: {s2_obs * factor:.4f} contra {s2_imp:.4f}")

print(f"n = {n}, observadas k = {k}, imputadas m = {n - k}")
print(f"varianza observada        s2_obs = {s2_obs:,.4f}")
print(f"varianza tras imputar            = {s2_imp:,.4f}")
print(f"factor teórico (k-1)/(n-1)       = {factor:.6f}")
print(f"s2_obs por el factor             = {s2_obs * factor:,.4f}   "
      f"(diferencia {abs(s2_obs * factor - s2_imp):.2e})")
print(f"\nel desvío cae de {np.sqrt(s2_obs):.4f} a {np.sqrt(s2_imp):.4f} "
      f"({100 * (1 - np.sqrt(s2_imp / s2_obs)):.2f} % menos), "
      f"con {100 * (n - k) / n:.2f} % de faltantes")

In [ ]:
# La forma que toma la distorsión, y las cuatro estrategias medidas. Celda dada.
observados = columna[~np.isnan(columna)]
imputada = np.where(np.isnan(columna), observados.mean(), columna)

fig, (izq, der) = plt.subplots(1, 2, figsize=(10.4, 3.6), sharey=True)
izq.hist(observados, bins=45, color=TURQUESA, alpha=0.85, label="valores observados")
izq.axvline(observados.mean(), color=NARANJA, lw=2, ls="--",
            label=f"media = {observados.mean():.1f}")
izq.set_xlabel("consumo_kwh")
izq.set_ylabel("Frecuencia")
izq.set_title("Antes de imputar")
izq.legend(fontsize=8.5)
der.hist(imputada, bins=45, color=TURQUESA, alpha=0.85)
der.axvline(observados.mean(), color=NARANJA, lw=2, ls="--")
der.set_xlabel("consumo_kwh")
der.set_title(f"Después de imputar la media: {n - k} valores idénticos")
der.annotate(f"desvío {np.sqrt(s2_obs):.2f} → {np.sqrt(s2_imp):.2f}",
             xy=(0.03, 0.88), xycoords="axes fraction", fontsize=9.5, color=NARANJA)
plt.tight_layout()
plt.show()

# Descartar filas es insesgado solo bajo MCAR: la composición por turno lo exhibe.
completo = df["turno"].value_counts(normalize=True)
print("composición por turno tras descartar las filas incompletas de cada columna:")
for col in ["consumo_kwh", "vibracion_mm_s"]:
    tras = df.loc[df[col].notna(), "turno"].value_counts(normalize=True)
    corr = " · ".join(f"{t} {100 * completo[t]:.2f} -> {100 * tras[t]:.2f} % "
                      f"({100 * (tras[t] - completo[t]):+.2f} pp)"
                      for t in ["mañana", "tarde", "noche"])
    print(f"  descartando por {col}:\n    {corr}")

# Las cuatro estrategias, sobre la misma partición y el mismo modelo.
A_num, B_num = ent[NUMERICAS].to_numpy(float), pru[NUMERICAS].to_numpy(float)


def con_indicador(A, B):
    """Agrega una columna binaria por cada atributo que tenga faltantes en A."""
    ia, ib = np.isnan(A).astype(float), np.isnan(B).astype(float)
    usar = ia.std(0) > 0
    return np.column_stack([A, ia[:, usar]]), np.column_stack([B, ib[:, usar]])


print("\nlas cuatro estrategias, medidas sobre el mismo modelo:")
for etiqueta, estrategia, indicador in [("media", "mean", False),
                                        ("mediana", "median", False),
                                        ("mediana + indicador de faltante", "median", True)]:
    Ai, Bi = con_indicador(A_num, B_num) if indicador else (A_num, B_num)
    im = SimpleImputer(strategy=estrategia).fit(Ai)
    mod = RandomForestClassifier(n_estimators=300, random_state=SEMILLA, n_jobs=-1).fit(
        im.transform(Ai), y_ent)
    print(f"  {etiqueta:<34} AUC = "
          f"{roc_auc_score(y_pru, mod.predict_proba(im.transform(Bi))[:, 1]):.4f}")

completas = ~np.isnan(A_num).any(1)
mod = RandomForestClassifier(n_estimators=300, random_state=SEMILLA, n_jobs=-1).fit(
    A_num[completas], y_ent[completas])
im = SimpleImputer(strategy="median").fit(A_num[completas])
print(f"  {'descartar las filas incompletas':<34} AUC = "
      f"{roc_auc_score(y_pru, mod.predict_proba(im.transform(B_num))[:, 1]):.4f}"
      f"   (se pierden {int((~completas).sum())} de {len(A_num)} filas de ajuste)")

### Para analizar

1. La identidad cierra con diferencia $0{,}00\times10^{0}$: la fórmula no aproxima, predice.
   Escribir qué procedimiento posterior —un contraste, un intervalo, un supuesto de
   normalidad— quedaría trabajando sobre una distribución fabricada, y por qué.
2. Descartar filas corre la composición del turno noche en más de seis puntos porcentuales al
   hacerlo por una columna, y en casi nada al hacerlo por la otra. Explicar la diferencia a
   partir del mecanismo diagnosticado en el ejercicio 4, y decir a qué población describe el
   modelo en cada caso.
3. Las cuatro estrategias difieren en **milésimas** de AUC. Corresponde decidir cuál va a
   producción y justificarlo **sin usar esa tabla**, porque la tabla no distingue nada.
   Nombrar el criterio que sí decide.

---

**Respuesta:**

1. Un contraste de medias, un intervalo de confianza o un modelo que estime errores usando esa varianza trabajaría sobre una distribución fabricada: la imputación agrega 259 observaciones exactamente en la media, crea una masa artificial y reduce la varianza de 14756.5483 a 13854.2948. Eso puede subestimar errores estándar, estrechar intervalos y falsear supuestos de forma o normalidad.
2. En `consumo_kwh`, compatible con MCAR, descartar completos conserva casi igual la composición por turno y el modelo describe aproximadamente la población original, aunque pierde precisión. En `vibracion_mm_s`, MAR respecto del turno, el descarte elimina desproporcionadamente casos nocturnos: noche pasa de 23.93 % a 17.82 %. Ese modelo describe la subpoblación con vibración observada, no la población objetivo completa.
3. Llevaría mediana más indicador de faltante: la mediana es robusta y el indicador preserva la información de ausencia relevante bajo MAR/MNAR. El criterio decisivo es el mecanismo de ausencia y la validez de los supuestos, no diferencias de milésimas de AUC en una sola partición. No descartaría filas porque al menos una ausencia no es MCAR.

## 5. El entregable

Todo lo anterior son mediciones. Lo que se entrega es una **decisión**, sostenida con las
cifras que este notebook produjo.

### DECIDE

Un área de mantenimiento pide un modelo que anticipe fallas para programar intervenciones
antes de que la máquina se detenga. Corresponde escribir la recomendación, y tiene que
responder las cinco preguntas:

1. **Qué columnas entran y qué columnas quedan afuera**, y por cada exclusión, cuál de las
   cuatro formas de fuga la motiva. `costo_reparacion_usd` e `id_maquina` tienen que estar
   nombradas explícitamente.
2. **Qué operaciones van antes de partir y qué operaciones van dentro del pipeline**, con el
   criterio que separa a unas de otras enunciado en una línea.
3. **Qué estrategia de faltantes** se adopta para cada una de las tres columnas, a partir del
   mecanismo diagnosticado y no del AUC.
4. **Qué número se reporta** como desempeño esperado, de todos los que el notebook produjo, y
   por qué ese y no otro. Decir también qué número **no** se puede reportar, y qué mediría en
   realidad.
5. **Qué supuesto queda declarado junto con el resultado.** Hay al menos uno que los datos no
   permiten verificar.

Una recomendación que no diga qué queda afuera y por qué está incompleta: el laboratorio
entero trata de decisiones que **no se ven en la métrica**.

**Respuesta:**

1. Entrarán `antiguedad_meses`, `horas_operacion`, `n_mantenimientos_previos`, `temperatura_c`, `vibracion_mm_s`, `presion_bar`, `consumo_kwh`, `planta`, `modelo_equipo` y `turno`. Quedan afuera `fallo`, por ser la respuesta; `id_registro`, por ser un identificador administrativo que además oculta duplicados; `costo_reparacion_usd`, por fuga de respuesta y temporal al registrarse después de la falla; e `id_maquina`, porque su codificación por la respuesta antes de validar fuga información del fold y, calculada correctamente, no aporta señal fuera de muestra (0.7842 con o sin ella).
2. Antes de partir van solo reglas que no estiman parámetros: convertir 999 y -1 a faltante, normalizar `modelo_equipo` y eliminar duplicados reales. Dentro del pipeline van imputación, escalado, one-hot y cualquier transformación aprendida. El criterio es: si la operación aprende algo de los datos, debe ajustarse exclusivamente con el bloque de entrenamiento de cada fold.
3. Para `consumo_kwh` (MCAR) usaré imputación por mediana dentro del pipeline; evita perder filas y, bajo MCAR, no necesita explotar la ausencia. Para `vibracion_mm_s` (MAR respecto del turno) usaré mediana más indicador de ausencia, conservando la señal del mecanismo observado; una mejora posterior sería una imputación condicional al turno. Para `temperatura_c` (MAR/MNAR no distinguible) usaré también mediana más indicador y acompañaré el resultado con un análisis de sensibilidad. Los centinelas de temperatura y presión se transforman antes en faltantes.
4. Reportaría AUC-VC = 0.7842 ± 0.0215, obtenido por validación cruzada estratificada de cinco folds que envuelve todo el pipeline limpio. Es la estimación que corresponde al flujo que se desplegaría. No reportaría 1.0000: mide la capacidad de explotar el costo registrado después de la falla, no de anticiparla. Tampoco usaría 0.8203, que mide una codificación de máquina contaminada con las respuestas de validación.
5. Declararía que el mecanismo de ausencia de `temperatura_c` no puede distinguirse entre MAR y MNAR con estos datos; el resultado supone que mediana más indicador transporta razonablemente ese mecanismo a producción. También declararía el supuesto de estabilidad/exchangeabilidad de la validación aleatoria: que la distribución futura de máquinas, turnos, sensores y faltantes será comparable a la observada. Hay que verificar externamente si el sensor de temperatura se satura y monitorear deriva después del despliegue.

---

## Para seguir explorando

Sin obligación y sin evaluación, para quien quiera seguir por su cuenta.

- **Las otras tres salidas para la alta cardinalidad.** La sección 2.2 midió la codificación
  por la respuesta. Quedan one-hot sobre los 180 niveles, la codificación por frecuencia y el
  agrupamiento de los niveles raros con un umbral. Medirlas sobre la misma validación cruzada
  y comparar el costo en columnas contra lo que aporta cada una. El umbral del agrupamiento es
  un hiperparámetro: conviene probar 10, 20 y 30 y ver cuántas filas cae en `Otros` con cada
  uno.
- **El escalador que se rompe.** Ajustar `StandardScaler` y `RobustScaler` sobre la columna
  `temperatura_c` **con los centinelas `999` adentro**, aplicarlos a las filas sanas, y medir
  qué media y qué desvío quedan. La estandarización debería dejar $0$ y $1$: comprobar cuánto
  se aparta, y explicar por qué el escalado robusto no se aparta.
- **El indicador de faltante como atributo predictivo.** Si la ausencia de temperatura es
  informativa, su indicadora tiene que aportar por sí sola. Ajustar un modelo usando
  **únicamente** las tres indicadoras de ausencia, sin ningún valor de sensor, y medir el AUC.
  Un resultado por encima del azar dice algo concreto sobre el mecanismo.
- **La imputación por modelo.** Completar cada faltante con la predicción de un modelo
  ajustado sobre las demás columnas. Dos advertencias antes de escribir una línea: es una
  transformación que **aprende de los datos**, con todo lo que eso implica sobre dónde tiene
  que vivir; y si el modelo imputador usa la respuesta, el resultado es una fuga de manual.